# BusinessMind RAG Evaluation — Simple Notebook

A compact notebook using **Groq + Hugging Face embeddings + FAISS + LangSmith**.

Place this notebook in your project root, beside `data/`. Add `GROQ_API_KEY` and `LANGSMITH_API_KEY` to a `.env` file. Keep keys private.

In [ ]:
%pip install -q langchain-groq langchain-community langchain-core langchain-text-splitters langchain-huggingface faiss-cpu sentence-transformers python-dotenv langsmith

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langsmith import Client
from langsmith.evaluation import evaluate

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")
DATASET_NAME = os.getenv("LANGSMITH_DATASET_NAME", "BusinessMind-RAG-Eval-v1")

assert GROQ_API_KEY, "Add GROQ_API_KEY to .env."
assert LANGSMITH_API_KEY, "Add LANGSMITH_API_KEY to .env."
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = os.getenv("LANGSMITH_PROJECT", "BusinessMind-RAG-Evaluation")
print("Configuration loaded. Groq model:", MODEL)

In [ ]:
# Load Markdown knowledge base and build FAISS once.
kb_dir = Path("data/knowledge_base")
files = sorted(kb_dir.glob("*.md"))
assert files, f"No Markdown files found in {kb_dir.resolve()}"

documents = []
for path in files:
    text = path.read_text(encoding="utf-8").strip()
    if text:
        documents.append(Document(page_content=text, metadata={"source": path.name}))

chunks = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100).split_documents(documents)
embeddings = HuggingFaceEmbeddings(model_name=os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"))
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})
llm = ChatGroq(model=MODEL, temperature=0, api_key=GROQ_API_KEY)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are BusinessMind's documentation assistant. Answer only from the retrieved context. Never invent policies, prices, deadlines, or account facts. If the context does not specify the answer, say so.\n\nContext:\n{context}"),
    ("human", "{question}")
])
answer_chain = prompt | llm
print(f"Loaded {len(documents)} documents and indexed {len(chunks)} chunks.")

In [ ]:
# RAG function used by the chatbot and evaluator.
def rag_app(inputs: dict) -> dict:
    question = inputs["question"]
    docs = retriever.invoke(question)
    context = "\n\n".join(
        f"[Source: {d.metadata.get('source', 'unknown')}]\n{d.page_content}"
        for d in docs
    )
    response = answer_chain.invoke({"question": question, "context": context})
    return {
        "answer": str(response.content),
        "documents": [
            {"source": d.metadata.get("source", "unknown"), "page_content": d.page_content}
            for d in docs
        ],
    }

# Test one question before evaluation.
rag_app({"question": "What products does BusinessMind offer?"})

In [ ]:
# Reuse the dataset if it already exists; otherwise create a small one.
client = Client(api_key=LANGSMITH_API_KEY)
existing = list(client.list_datasets(dataset_name=DATASET_NAME))
if existing:
    dataset = existing[0]
    print("Reusing dataset:", dataset.name)
else:
    dataset = client.create_dataset(dataset_name=DATASET_NAME, description="BusinessMind RAG evaluation")
    examples = [
        {"inputs": {"question": "What products does BusinessMind offer?"},
         "outputs": {"answer": "ClientSphere provides customer relationship management and sales-pipeline tracking. SecureSync provides secure file synchronization and team file sharing."}},
        {"inputs": {"question": "What is BusinessMind's return policy?"},
         "outputs": {"answer": "There is no universal return or refund policy. Eligibility depends on contract and order terms, subscription status, and applicable law. Customers should contact support with their order reference."}},
    ]
    client.create_examples(dataset_id=dataset.id, examples=examples)
    print("Created dataset:", dataset.name)
print("Dataset ID:", dataset.id)

In [ ]:
# Four small LLM-as-judge metrics.
def judge(key, instruction, inputs, outputs, reference_outputs=None):
    data = {
        "question": inputs.get("question", ""),
        "answer": outputs.get("answer", ""),
        "documents": outputs.get("documents", []),
    }
    if reference_outputs:
        data["reference_answer"] = reference_outputs.get("answer", "")
    response = llm.invoke([
        ("system", 'Return only valid JSON: {"score": true, "explanation": "short reason"}. The score must be a boolean.'),
        ("human", instruction + "\n\n" + json.dumps(data, ensure_ascii=False)),
    ])
    raw = str(response.content).strip().replace("```json", "").replace("```", "").strip()
    result = json.loads(raw)
    return {"key": key, "score": result["score"], "comment": result.get("explanation", "")}

def correctness(inputs, outputs, reference_outputs):
    return judge("correctness", "Is the answer factually consistent with the reference answer?", inputs, outputs, reference_outputs)

def groundedness(inputs, outputs, reference_outputs=None):
    return judge("groundedness", "Are the answer's factual claims supported by the retrieved documents?", inputs, outputs, reference_outputs)

def relevance(inputs, outputs, reference_outputs=None):
    return judge("relevance", "Does the answer directly address the question?", inputs, outputs, reference_outputs)

def retrieval_relevance(inputs, outputs, reference_outputs=None):
    return judge("retrieval_relevance", "Do the retrieved documents contain useful evidence for answering the question?", inputs, outputs, reference_outputs)

print("All four evaluators are ready.")

In [ ]:
# Run evaluation and inspect results in LangSmith.
results = evaluate(
    rag_app,
    data=DATASET_NAME,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="businessmind-rag-simple",
    metadata={"model": MODEL},
)
print("Evaluation submitted. Open your LangSmith project to review scores.")